# Alexandria Lint/Split Smoke Notebook

This notebook is a manual smoke check for the split-check lifecycle. It uses the configured Postgres database, real SQL repositories, real unit of work, real outbox repository, and the worker handling boundary.

Only the `Splitter` is fake. That keeps the split plan deterministic while still exercising durable infrastructure and application split validation.


## Prerequisites

Expected setup:
- Docker and Task are available locally.
- The configured Postgres database is reachable through `.env` or the environment.
- The notebook is run from the repository root or from the `sandbox/` directory.

Provider credentials are not required for this smoke path because the fixture rows are inserted directly and the fake splitter supplies the split plan. The app may still construct configured adapters, but this notebook does not call embedding or summarization providers.


## Start Infrastructure

This cell starts the local infrastructure profile through the project Taskfile and waits for Docker Compose health checks before the smoke path runs.


In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess


repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent

deploy = subprocess.run(
    ["task", "deploy", "--", "--wait"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

if deploy.stdout:
    print(deploy.stdout)
if deploy.stderr:
    print(deploy.stderr)


## Configure A Deterministic Splitter

The fake splitter is the only fake in this notebook. It returns a stable two-child plan so the smoke check can assert exact document movement after the real split use case validates the plan.


In [ ]:
from typing import Any

from application.ports import ChildPlan, SplitPlan
from domain.entity import VECTOR_DIMENSIONS


def vector(*values: float) -> list[float]:
    embedding = [0.0] * VECTOR_DIMENSIONS
    for index, value in enumerate(values):
        embedding[index] = value
    return embedding


class DeterministicSplitter:
    def __init__(self) -> None:
        self.calls: list[tuple[object, list[object]]] = []

    async def split(self, node: Any, docs: list[Any]) -> SplitPlan:
        ordered = sorted(docs, key=lambda item: item.name)
        self.calls.append((node.id, [doc.id for doc in ordered]))
        return SplitPlan(
            children=[
                ChildPlan(
                    name="Alpha child",
                    description="Documents about alpha topics.",
                    embedding=vector(0.1),
                    docs=[ordered[0].id, ordered[2].id],
                ),
                ChildPlan(
                    name="Beta child",
                    description="Documents about beta topics.",
                    embedding=vector(0.2),
                    docs=[ordered[1].id],
                ),
            ]
        )


## Build Durable Fixture State

This cell creates a full active leaf, three documents, a stale outgoing reference, and a `split.check` job in the configured Postgres database. It uses the real schema, real SQLAlchemy models, and real outbox repository.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

from sqlalchemy import select

from domain.entity import Document, Job, Node, Reference
from domain.values import JobKind
from infrastructure.config import IngestSettings, get_settings
from infrastructure.persistence.db import Db
from infrastructure.repositories.outbox import OutboxRepo


get_settings.cache_clear()
base_settings = get_settings()
settings = base_settings.model_copy(update={"ingest": IngestSettings(max_leaf_docs=2)})
db = Db(settings.database)
db.create_all()

stamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
source_id = uuid4()
target_id = uuid4()
doc_ids = [uuid4(), uuid4(), uuid4()]
ref_id = uuid4()

with db.session() as session:
    source = Node(
        id=source_id,
        name=f"Notebook Source {stamp}",
        description="Full source leaf for the lint/split smoke notebook.",
        embedding=vector(1.0),
        kind="leaf",
        status="active",
        doc_count=3,
    )
    target = Node(
        id=target_id,
        name=f"Notebook Target {stamp}",
        description="Referenced target leaf for stale-reference cleanup.",
        embedding=vector(2.0),
        kind="leaf",
        status="active",
    )
    docs = [
        Document(
            id=doc_ids[0],
            leaf=source,
            source_key=f"notebook:lint-split:alpha:{stamp}",
            name="Alpha",
            summary="A",
            body="A",
            embedding=vector(10.0),
        ),
        Document(
            id=doc_ids[1],
            leaf=source,
            source_key=f"notebook:lint-split:beta:{stamp}",
            name="Beta",
            summary="B",
            body="B",
            embedding=vector(11.0),
        ),
        Document(
            id=doc_ids[2],
            leaf=source,
            source_key=f"notebook:lint-split:gamma:{stamp}",
            name="Gamma",
            summary="G",
            body="G",
            embedding=vector(12.0),
        ),
    ]
    ref = Reference(id=ref_id, from_node=source, to_node=target, distance=0.2, rank=0)
    session.add_all([source, target, *docs, ref])
    session.flush()

    outbox = OutboxRepo(session, settings.queue)
    job_id = await outbox.append(
        Job(
            kind=JobKind.SPLIT_CHECK,
            payload={"node_id": str(source.id)},
            key=source.id,
        )
    )
    session.commit()

print("source_id", source_id)
print("target_id", target_id)
print("job_id", job_id)
print("doc_ids", [str(doc_id) for doc_id in doc_ids])


## Inspect Before State

Expected notes:
- The source node should be an active leaf with three documents.
- The stale outgoing reference should exist before splitting.
- The queued job should be pending before worker handling.


In [ ]:
with db.session() as session:
    source = session.get(Node, source_id)
    assert source is not None
    source_docs = session.scalars(
        select(Document).where(Document.leaf_id == source_id).order_by(Document.name.asc())
    ).all()
    refs = session.scalars(select(Reference).where(Reference.from_node_id == source_id)).all()
    job = session.get(Job, job_id)
    assert job is not None

    print("before")
    print("  source:", source.kind, source.status, source.doc_count)
    print("  docs:", [doc.name for doc in source_docs])
    print("  outgoing_refs:", len(refs))
    print("  job:", job.status, job.attempts, job.payload)

    assert source.kind == "leaf"
    assert source.status == "active"
    assert len(source_docs) == 3
    assert len(refs) == 1
    assert job.status == "pending"


## Handle The Split-Check Job

This cell constructs the real `App` with the fake splitter injected, marks the known outbox job as running, sends that exact job through the worker handler, and marks it done through the real outbox repository.

It intentionally handles the known job id instead of calling an unscoped worker batch, so the notebook does not process unrelated pending jobs in a developer database.


In [ ]:
from datetime import datetime, timezone

from application.app import App
from domain.values import JobKind, JobStatus
from infrastructure.repositories.outbox import OutboxRepo
from presentation.worker.app import Worker


splitter = DeterministicSplitter()
app = App(settings, splitter=splitter)
worker = Worker(app=app, kind=JobKind.SPLIT_CHECK)

with db.session() as session:
    claimed = session.get(Job, job_id)
    assert claimed is not None
    claimed.status = JobStatus.RUNNING.value
    claimed.locked_at = datetime.now(timezone.utc)
    claimed.attempts += 1
    session.flush()
    worker_job = Job(id=claimed.id, kind=claimed.kind, payload=dict(claimed.payload), key=claimed.key)
    session.commit()

try:
    await worker.handle(worker_job)
except Exception as error:
    with db.session() as session:
        outbox = OutboxRepo(session, settings.queue)
        await outbox.mark(job_id, JobStatus.FAILED, str(error))
        session.commit()
    raise
else:
    with db.session() as session:
        outbox = OutboxRepo(session, settings.queue)
        await outbox.mark(job_id, JobStatus.DONE)
        session.commit()

print("processed job", job_id)
print("splitter calls", [(str(node_id), [str(doc_id) for doc_id in docs]) for node_id, docs in splitter.calls])


## Inspect After State

Expected notes:
- The source leaf should become an active branch.
- Two active child leaves should exist under the source node.
- Documents should move to the child leaves according to the fake split plan.
- The stale outgoing reference from the source should be cleared.
- The outbox job should be marked done.


In [ ]:
from domain.values import JobStatus


with db.session() as session:
    source = session.get(Node, source_id)
    assert source is not None
    children = session.scalars(
        select(Node).where(Node.parent_id == source_id).order_by(Node.name.asc())
    ).all()
    docs = session.scalars(
        select(Document).where(Document.id.in_(doc_ids)).order_by(Document.name.asc())
    ).all()
    refs = session.scalars(select(Reference).where(Reference.from_node_id == source_id)).all()
    job = session.get(Job, job_id)
    assert job is not None

    child_by_id = {child.id: child for child in children}
    assignments = {doc.name: child_by_id[doc.leaf_id].name for doc in docs}
    child_counts = {child.name: child.doc_count for child in children}

    print("source", source.kind, source.status, source.doc_count, source.version)
    print("children", [(child.name, child.doc_count) for child in children])
    print("docs", [(doc.name, assignments[doc.name]) for doc in docs])
    print("outgoing_refs", len(refs))
    print("job", job.status, job.done_at is not None)

    assert source.kind == "branch"
    assert source.status == "active"
    assert source.doc_count == 0
    assert child_counts == {"Alpha child": 2, "Beta child": 1}
    assert assignments == {
        "Alpha": "Alpha child",
        "Beta": "Beta child",
        "Gamma": "Alpha child",
    }
    assert len(refs) == 0
    assert job.status == JobStatus.DONE.value
    assert job.done_at is not None


### Expected Smoke Output

- `before` should show a full active leaf, three documents, one outgoing reference, and a pending job.
- `processed job` should print the known outbox job id.
- `after` should show one source branch, two child leaves, moved documents, zero outgoing refs, and a done job.
- The fake splitter call should include the source node id and the three source document ids.


### Matching Pytest Smoke Command

Run this from the repository root:

```bash
uv run pytest tests/integration/test_lint_split_flow.py -q
```


## Tear Down Infrastructure

Run this final cell when the smoke check is complete. It stops the local infrastructure profile through the project Taskfile without deleting Docker volumes.


In [ ]:
from pathlib import Path
import subprocess


repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent

teardown = subprocess.run(
    ["task", "teardown"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

if teardown.stdout:
    print(teardown.stdout)
if teardown.stderr:
    print(teardown.stderr)
